In [2]:
from dataset_setup import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(batch_size=64)

Using downloaded and verified file: C:\Users\shali\.medmnist\octmnist_224.npz
Using downloaded and verified file: C:\Users\shali\.medmnist\octmnist_224.npz
Using downloaded and verified file: C:\Users\shali\.medmnist\octmnist_224.npz


In [9]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_feature_extractor():
    model = models.resnet18(weights='IMAGENET1K_V1')
    #un solo canale
    model.conv1 = nn.Conv2d(1, 64, kernel_size =7, stride=2, padding=3, bias =False)
    model.fc = nn.Identity()
    #FREEZE
    for param in model.parameters():
        param.requires_grad = False
    return model

# --- Seed ---
CURRENT_SEED = 79  # 11, 17, 79
# ------------

def main():
    set_seed(CURRENT_SEED)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = get_feature_extractor().to(device)
    model.eval()

    features_list = []
    labels_list = []

    print("Estrazione in corso...")
    with torch.no_grad():
        for images, labels in train_loader:
            images = images.to(device)
            output = model(images)
            
            features_list.append(output.cpu())
            labels_list.append(labels.cpu())

    final_features = torch.cat(features_list, dim=0)
    final_labels = torch.cat(labels_list, dim=0)

    print(f"Completato! Shape finale: {final_features.shape}")
    
    data_to_save = {
        'features': final_features,
        'labels': final_labels,     
        'seed': CURRENT_SEED,
        'model': 'resnet18_frozen'
    }
    
    torch.save(data_to_save, f"../extractedFeatures/extracted_features_seed_{CURRENT_SEED}.pt")
    print(f"Dataset salvato correttamente per il seed {CURRENT_SEED}")

if __name__ == "__main__":
    main()

Estrazione in corso...
Completato! Shape finale: torch.Size([97477, 512])
Dataset salvato correttamente per il seed 79
